# PaliGemma Image Analysis API - Colab

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gómez Soteres  
**Universidad Francisco de Vitoria**

---

This notebook exposes a **FastAPI endpoint** for analyzing campus navigation images with PaliGemma.

## Purpose:
- Load PaliGemma model (`google/paligemma2-3b-mix-224`) on Colab GPU
- Expose REST API endpoint `/analyze` for image analysis
- Use ngrok to make the API publicly accessible
- Integrate with local FastAPI server for campus navigation planning

## Features:
- **GPU Acceleration**: Uses Colab's free GPU (T4/V100)
- **FastAPI**: Modern async API framework
- **Public Access**: ngrok tunnel for external access
- **Image Analysis**: Reads campus signage to identify building and floor

## Usage:
1. Run all cells in order
2. Copy the ngrok URL provided
3. Set `COLAB_API_URL` in `src/api/planning_api.py`
4. Your local API will use Colab's GPU for image analysis

## 1️. Setup and Installation

In [ ]:
# Install required packages
!pip install -q fastapi uvicorn python-multipart
!pip install -q transformers>=4.40.0 accelerate
!pip install -q torch torchvision
!pip install -q Pillow
!pip install -q pyngrok
!pip install -q openai-whisper

print("All packages installed successfully!")

## 2️. Clone Repository from GitHub

In [ ]:
# Setup project - Clone from GitHub (Private Repo)
import os
import sys
import subprocess

# ============================================================
# CONFIGURATION - PRIVATE REPO
# ============================================================
GITHUB_USER = "AlbertoGomezS"
GITHUB_REPO = "tfg-AlbertoGomez"
LOCAL_PROJECT_PATH = '/content/tfg'

# GitHub Personal Access Token (create at: https://github.com/settings/tokens)
# Required scopes: repo (full control of private repositories)
GITHUB_TOKEN = "***REMOVED***"  # <-- REPLACE THIS

# ============================================================

# Clone or update from GitHub
if os.path.exists(LOCAL_PROJECT_PATH):
    # Update existing repo
    os.chdir(LOCAL_PROJECT_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("Repository updated from GitHub")
    else:
        print(f"Git pull issue: {result.stderr}")
else:
    # Clone with authentication (private repo)
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    result = subprocess.run(['git', 'clone', clone_url, LOCAL_PROJECT_PATH], capture_output=True, text=True)

    if result.returncode == 0:
        os.chdir(LOCAL_PROJECT_PATH)
        print(f"Repository cloned from GitHub (private)")
    else:
        print(f"Clone failed. Check your GITHUB_TOKEN")
        print(f"   Error: {result.stderr}")
        raise Exception("Could not clone repository - check token")

# Add src to path
sys.path.insert(0, LOCAL_PROJECT_PATH)

print(f"Working directory: {os.getcwd()}")

## 3️. HuggingFace Authentication

In [ ]:
# Login to HuggingFace (required for PaliGemma access)
from huggingface_hub import login

# ============================================================
# CONFIGURATION - HUGGINGFACE TOKEN
# ============================================================
# IMPORTANT: Replace with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the PaliGemma license

HF_TOKEN = "***REMOVED***"  # <-- REPLACE THIS

# ============================================================

if HF_TOKEN != "***REMOVED***":
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace")
else:
    print("Please set your HuggingFace token above")
    print("   1. Get your token from: https://huggingface.co/settings/tokens")
    print("   2. Accept the PaliGemma license: https://huggingface.co/google/paligemma2-3b-mix-224")
    print("   3. Replace 'HF_TOKEN' with your actual token")

## 4️. Load PaliGemma Model

In [ ]:
## 4.5. Load Whisper Model

import whisper
import torch

# Check device (needed for GPU memory reporting)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Whisper model (base model is good balance between speed and accuracy)
# Options: tiny, base, small, medium, large
whisper_model = whisper.load_model("base")
print("Whisper model loaded successfully!")

# Show GPU memory usage after loading Whisper
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU Memory after Whisper: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

In [ ]:
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import torch

# Model configuration
model_id = "google/paligemma2-3b-mix-224"

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

print(f"\nLoading model: {model_id}")
print("   This may take a few minutes...")

# Load processor
processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)

# Load model
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else device,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully!")

# Show GPU memory usage
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\nGPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

## 5️. Create FastAPI Application

In [ ]:
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io
import uvicorn

# Create FastAPI app
app = FastAPI(
    title="PaliGemma Image Analysis API",
    description="API for analyzing campus navigation images with PaliGemma",
    version="1.0.0"
)

# CORS middleware - Allow all origins (for local development)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Prompt for PaliGemma (default - in English)
# This is the first prompt from the multiple prompts strategy
PROMPT = "Read the signage board in this campus image. Identify the building name and floor number displayed on the sign. Look for words like 'BUILDING' or 'FLOOR' and the associated text."

@app.get("/")
async def root():
    """Root endpoint - API status."""
    return {
        "message": "PaliGemma Image Analysis API",
        "status": "running",
        "model": model_id,
        "device": device
    }

@app.post("/analyze")
async def analyze_image(file: UploadFile = File(...), prompt: str = None):
    """
    Analyze an image with PaliGemma to extract building and floor information.
    
    Args:
        file: Image file (JPEG, PNG)
        prompt: Optional custom prompt. If not provided, uses default PROMPT.
    
    Returns:
        JSON with 'success', 'response' (PaliGemma output), and 'error'
    """
    try:
        # Validate file type
        if file.content_type not in ["image/jpeg", "image/jpg", "image/png"]:
            return {
                "success": False,
                "response": "",
                "error": "Formato de imagen no soportado. Use JPEG o PNG."
            }
        
        # Use custom prompt if provided, otherwise use default
        prompt_to_use = prompt if prompt else PROMPT
        
        # Read image
        image_data = await file.read()
        image = Image.open(io.BytesIO(image_data)).convert("RGB")
        
        # Prepare inputs for PaliGemma
        inputs = processor(
            text=prompt_to_use,
            images=image,
            padding="longest",
            do_convert_rgb=True,
            return_tensors="pt"
        ).to(model.device)
        
        inputs = inputs.to(dtype=model.dtype)
        
        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                num_beams=1,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
        
        # Decode response
        input_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][input_len:]
        response = processor.decode(generated_ids, skip_special_tokens=True).strip()
        
        return {
            "success": True,
            "response": response,
            "error": None
        }
    
    except Exception as e:
        import traceback
        error_msg = str(e)
        print(f"Error: {error_msg}")
        traceback.print_exc()
        return {
            "success": False,
            "response": "",
            "error": error_msg
        }

@app.post("/transcribe")
async def transcribe_audio(file: UploadFile = File(...)):
    """
    Transcribe audio to text using Whisper.
    
    Args:
        file: Audio file (WAV, MP3, M4A, etc.)
    
    Returns:
        JSON with 'success', 'text' (transcribed text), and 'error'
    """
    try:
        # Validate file type
        if not file.content_type or not any(x in file.content_type for x in ["audio", "video"]):
            return {
                "success": False,
                "text": "",
                "error": "Formato de audio no soportado. Use WAV, MP3, M4A, etc."
            }
        
        # Read audio file
        audio_data = await file.read()
        
        # Save to temporary file (Whisper needs a file path)
        import tempfile
        import os
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp_file:
            tmp_file.write(audio_data)
            tmp_path = tmp_file.name
        
        try:
            # Transcribe with Whisper (English)
            result = whisper_model.transcribe(tmp_path, language="en")
            transcribed_text = result["text"].strip()
            
            return {
                "success": True,
                "text": transcribed_text,
                "error": None
            }
        finally:
            # Clean up temporary file
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
    
    except Exception as e:
        import traceback
        error_msg = str(e)
        print(f"Error transcribing: {error_msg}")
        traceback.print_exc()
        return {
            "success": False,
            "text": "",
            "error": error_msg
        }


print("FastAPI application created!")

In [ ]:
from pyngrok import ngrok
import socket

# ============================================================
# CONFIGURATION - NGROK AUTHTOKEN
# ============================================================
# Get your free ngrok authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
# Then run: ngrok config add-authtoken YOUR_TOKEN
# Or set it here:

NGROK_AUTHTOKEN = None  # <-- OPCIONAL: Poner tu token aquí si prefieres

# ============================================================

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    print("ngrok authtoken configurado")
else:
    print("Si es la primera vez, configura ngrok:")
    print("   1. Ve a: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   2. Copia tu authtoken")
    print("   3. Ejecuta: !ngrok config add-authtoken TU_TOKEN")
    print("   O ponlo en NGROK_AUTHTOKEN arriba")

# Encontrar puerto disponible
def find_free_port(start_port=8001):
    for port in range(start_port, start_port + 10):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('0.0.0.0', port))
                return port
            except OSError:
                continue
    return 8001  # Fallback

PORT = find_free_port(8001)
print(f"\nUsando puerto: {PORT}")

# Start server in background FIRST
import threading
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

print("\nIniciando servidor FastAPI...")
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to be ready
print("   Esperando a que el servidor esté listo...")
time.sleep(3)  # Give server time to start

# Kill any existing ngrok tunnels before creating a new one
print("   Cerrando túneles ngrok existentes...")
try:
    # Method 1: Try to disconnect all tunnels from this pyngrok instance
    tunnels = ngrok.get_tunnels()
    if tunnels:
        print(f"   Encontrados {len(tunnels)} túnel(es) activo(s) en esta instancia")
        for tunnel in tunnels:
            try:
                ngrok.disconnect(tunnel.public_url)
                print(f"   Túnel cerrado: {tunnel.public_url}")
            except Exception as e:
                print(f"   Error cerrando túnel {tunnel.public_url}: {e}")
    
    # Method 2: Kill all pyngrok processes
    ngrok.kill()
    
    # Method 3: Try to use ngrok API to close any active tunnels
    try:
        import requests
        import time as time_module
        # Wait a bit for ngrok to start if it was just killed
        time_module.sleep(1)
        
        # Try to access ngrok local API
        ngrok_api = "http://127.0.0.1:4040/api/tunnels"
        try:
            response = requests.get(ngrok_api, timeout=3)
            if response.status_code == 200:
                tunnels_data = response.json()
                for tunnel in tunnels_data.get("tunnels", []):
                    tunnel_name = tunnel.get("name", "")
                    if tunnel_name:
                        disconnect_url = f"http://127.0.0.1:4040/api/tunnels/{tunnel_name}"
                        delete_response = requests.delete(disconnect_url, timeout=3)
                        if delete_response.status_code in [204, 200]:
                            print(f"   Túnel cerrado vía API: {tunnel.get('public_url', '')}")
        except requests.exceptions.ConnectionError:
            # ngrok API not accessible, which is fine
            pass
        except Exception as e:
            print(f"   (No se pudo acceder a API de ngrok: {e})")
    except Exception as e:
        pass  # If requests is not available, skip
    
except Exception as e:
    print(f"   (Algunos túneles pueden seguir activos: {e})")

# Small delay to ensure tunnels are closed
time.sleep(2)

# Now connect ngrok
print("   Conectando con ngrok...")
try:
    public_url = ngrok.connect(PORT)
    api_url = f"{public_url.public_url}"
except Exception as e:
    # If connection fails due to existing tunnel, try with a different approach
    error_str = str(e)
    if "already online" in error_str or "ERR_NGROK_334" in error_str:
        print("   ⚠️ El túnel anterior sigue activo. Intentando forzar nuevo túnel...")
        # Wait a bit more and try again
        time.sleep(3)
        try:
            public_url = ngrok.connect(PORT)
            api_url = f"{public_url.public_url}"
        except Exception as e2:
            print(f"   ❌ Error: {e2}")
            print("\n   💡 SOLUCIÓN: Ve a https://dashboard.ngrok.com/status/tunnels")
            print("      y cierra manualmente el túnel activo, luego vuelve a ejecutar esta celda.")
            raise e2
    else:
        raise e

print("\n" + "="*60)
print("API PUBLICA DISPONIBLE")
print("="*60)
print(f"\nURL publica: {api_url}")
print(f"Endpoint de analisis: {api_url}/analyze")
print(f"\nIMPORTANTE: Copia esta URL y usala en tu codigo local")
print(f"   En 'src/api/planning_api.py', configura:")
print(f"   COLAB_API_URL = '{api_url}'")
print("\n" + "="*60)

print("\nServidor iniciado en segundo plano")
print(f"   Puerto: {PORT}")
print("   El servidor seguirá corriendo mientras esta celda esté activa")
print("   Para detener: Interrumpir ejecución o reiniciar runtime")

## 7️. Test the API (Optional)

In [ ]:
# Test endpoint (optional)
import requests

# Test root endpoint
try:
    response = requests.get(f"{api_url}/")
    print("Root endpoint test:")
    print(response.json())
except Exception as e:
    print(f"Error: {e}")

# Note: To test /analyze, you need to upload an image file
# You can test it from your local machine using the URL above

## Notes

- **Keep this notebook running**: The API will be available as long as this notebook is active
- **Ngrok URL changes**: Each time you restart ngrok, you'll get a new URL
- **Update local code**: Remember to update `COLAB_API_URL` in `planning_api.py` if the URL changes
- **GPU Usage**: Colab provides free GPU, but sessions may timeout after inactivity
- **Model Loading**: First load takes time, subsequent requests are fast